# Can sequential models beat CatBoost? Beijing PM2.5 forecasting

**Reproducible experiment:** predict the next **24 hours** of PM2.5 concentration at Beijing's **Aotizhongxin** monitoring station from **168 past hours** of pollution and weather measurements. Switch to `HORIZON = 168` for a one-week forecast. Compare **MLP, GRU, LSTM, TCN, Transformer, CatBoost (raw), CatBoost (engineered)** and seasonal-naive baselines.

> This is a *new benchmark*, not a reproduction of the published paper's exact protocol, and there is **no guarantee** that a sequential model wins. Train/validation/test partitions are strictly chronological. All models predict exactly the same held-out forecast origins.

**Dataset:** [Beijing Multi-Site Air Quality — UCI Repository](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data). Data collected March 2013–February 2017; 12 stations; 420,768 station-hour records. Citation: Chen (2017), doi:10.24432/C5RK5G. Licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Important:** Real dataset needs internet access once, or provide the downloaded ZIP/CSV using the path setting below. A notebook with no saved model scores is intentional: run the cells to obtain your own measured comparison.

## 1. Setup and experiment switches

Install dependencies once (`pip install numpy pandas matplotlib scikit-learn torch catboost`). QUICK mode is a credible *smoke / exploratory* comparison, but the full mode is recommended for conclusions; tuning should use only validation data.

In [ ]:
# Uncomment if needed, then restart the kernel:
# %pip install -U numpy pandas matplotlib scikit-learn torch catboost

import io, zipfile, time, random, copy, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from catboost import CatBoostRegressor

SEED = 42
STATION = 'Aotizhongxin'
TARGET = 'PM2.5'
LOOKBACK = 168             # past hours
HORIZON = 24              # set to 168 for a 1-week forecast
QUICK_RUN = True          # False = stronger benchmark; takes longer
NEURAL_USE_ENGINEERED = False  # optional pipeline comparison
RUN_MODELS = ['MLP', 'GRU', 'LSTM', 'TCN', 'Transformer']
DATA_PATH = None          # optional local UCI .zip, extracted folder, or station .csv
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TRAIN_ORIGIN_STRIDE = 12 if QUICK_RUN else 3
EVAL_ORIGIN_STRIDE = 24
MAX_EPOCHS = 18 if QUICK_RUN else 100
PATIENCE = 5 if QUICK_RUN else 14
CB_ITERATIONS = 140 if QUICK_RUN else 650
CB_EARLY_STOPPING = 25 if QUICK_RUN else 60
BATCH_SIZE = 96 if QUICK_RUN else 128
LEARNING_RATE = 1e-3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
print(f'{DEVICE=}  {LOOKBACK=}  {HORIZON=}  {QUICK_RUN=}  {CB_ITERATIONS=}')

## 2. Download and load the real UCI measurements

The UCI download is usually a ZIP containing a nested ZIP. This loader handles nested archives, direct station CSVs, and extracted folders. If the automatic download is blocked, download [the original UCI archive](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data), set `DATA_PATH` to its path, and rerun.

In [ ]:
UCI_URL = 'https://archive.ics.uci.edu/static/public/501/beijing%2Bmulti%2Bsite%2Bair%2Bquality%2Bdata.zip'
GITHUB_STATION_URL = ('https://raw.githubusercontent.com/KM-drago/'
                      'Beijing-Multi-Site-Air-Quality-Data/main/'
                      f'PRSA_Data_{STATION}_20130301-20170228.csv')
CACHE_DIR = Path('beijing_air_quality_cache')
CACHE_DIR.mkdir(exist_ok=True)
STATION_CSV = f'PRSA_Data_{STATION}_20130301-20170228.csv'


def csv_from_archive(blob, station, max_depth=3):
    if max_depth <= 0:
        raise ValueError('Nested archive too deep')
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        files = zf.namelist()
        candidates = [p for p in files if p.lower().endswith('.csv') and station.lower() in p.lower()]
        if candidates:
            with zf.open(candidates[0]) as f:
                return pd.read_csv(f)
        for p in files:
            if p.lower().endswith('.zip'):
                with zf.open(p) as f:
                    try:
                        return csv_from_archive(f.read(), station, max_depth-1)
                    except (ValueError, zipfile.BadZipFile):
                        pass
    raise FileNotFoundError(f'Could not find station {station} in archive')


def download_bytes(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (forecasting research notebook)'})
    with urllib.request.urlopen(req, timeout=45) as response:
        return response.read()


def load_station(station=STATION, data_path=DATA_PATH):
    local = Path(data_path) if data_path is not None else None
    if local is not None:
        if local.is_dir():
            matches = list(local.rglob(f'*{station}*.csv'))
            if not matches:
                raise FileNotFoundError(f'No CSV for {station} under {local}')
            return pd.read_csv(matches[0])
        if local.suffix.lower() == '.csv':
            return pd.read_csv(local)
        if local.suffix.lower() == '.zip':
            return csv_from_archive(local.read_bytes(), station)
        raise ValueError('DATA_PATH must be ZIP, CSV, or extracted directory')
    station_cache = CACHE_DIR / STATION_CSV
    if station_cache.exists():
        return pd.read_csv(station_cache)
    errors=[]
    for url, is_zip in [(UCI_URL, True), (GITHUB_STATION_URL, False)]:
        try:
            print('Downloading:', url)
            payload=download_bytes(url)
            raw=csv_from_archive(payload, station) if is_zip else pd.read_csv(io.BytesIO(payload))
            if not {'PM2.5', 'year', 'month', 'day', 'hour'}.issubset(raw.columns):
                raise ValueError('Unexpected CSV schema')
            raw.to_csv(station_cache, index=False)
            return raw
        except Exception as exc:
            errors.append(f'{url}: {type(exc).__name__}({exc})')
    raise RuntimeError('Automatic downloads failed. Download the UCI ZIP and set DATA_PATH.\n'+'\n'.join(errors))

raw = load_station()
print('Raw station rows:', len(raw))
print('Columns:', list(raw.columns))
display(raw.head())

## 3. Prepare a strictly time-ordered series

**No lookahead:** missing sensor values are replaced only with the last *earlier* observed value (`ffill`); leading missing values use medians computed from **training dates only**. The unfilled PM2.5 observation mask is retained to evaluate real measurements, not imputed test targets. Wind direction is converted to sine/cosine components, and no future weather observations are provided.

In [ ]:
required = ['PM2.5','PM10','SO2','NO2','CO','O3','TEMP','PRES','DEWP','RAIN','WSPM']
missing = set(required)-set(raw.columns)
assert not missing, f'Missing required columns: {missing}'

frame = raw.copy()
frame.index = pd.to_datetime(frame[['year','month','day','hour']])
frame = frame.sort_index()
assert frame.index.is_unique, 'Duplicated datetimes in selected station'
frame = frame.reindex(pd.date_range(frame.index.min(), frame.index.max(), freq='h'))

wind_dirs=['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW']
angles = {w: i * 2*np.pi/16 for i,w in enumerate(wind_dirs)}
wind = frame['wd'].map(angles)
frame['wind_sin']=np.sin(wind)
frame['wind_cos']=np.cos(wind)
feature_cols = required + ['wind_sin','wind_cos']
for column in feature_cols:
    frame[column] = pd.to_numeric(frame[column], errors='coerce')

observed_pm25 = frame[TARGET].notna().to_numpy()  # BEFORE imputation
train_end = int(len(frame)*0.70)
val_end = int(len(frame)*0.85)
train_medians = frame.iloc[:train_end][feature_cols].median().fillna(0)
frame[feature_cols] = frame[feature_cols].ffill().fillna(train_medians)
assert not frame[feature_cols].isna().any().any()

print('Date range:',frame.index.min(), 'to', frame.index.max())
print('Training ends:', frame.index[train_end], '| validation ends:',frame.index[val_end])
print('Observed PM2.5 fraction:', round(observed_pm25.mean(),4))
fig, ax=plt.subplots(figsize=(12,3))
frame[TARGET].iloc[::6].plot(ax=ax, lw=.6, title=f'{STATION} PM2.5 (causally filled for model inputs)')
ax.set_ylabel('µg/m³'); plt.tight_layout(); plt.show()

## 4. Build forecasting windows and leakage-safe statistical features

Each forecast origin `t` has inputs from `[t-168, ..., t-1]`, outputs from `[t, ..., t+H-1]`, plus **known future calendar** for those H hours (hour-of-day / day-of-week cycles). No observed values at `t` or later enter either CatBoost or the neural models.

Raw CatBoost sees the flattened history + future calendar. Engineered CatBoost additionally gets **past-only lag values, rolling means/std/min/max, 12–168 hour slopes, yesterday/week-ago differences and historical weather summaries**. All neural models see the same original information; set `NEURAL_USE_ENGINEERED=True` to test engineered inputs for them too.

Training forecasts end before validation begins; validation forecasts end before test begins. Past context may cross these boundaries—this is realistic when forecasting newly arriving data.

In [ ]:
def future_calendar(index_slice):
    hours=np.asarray(index_slice.hour, dtype=np.float64)
    weekdays=np.asarray(index_slice.dayofweek, dtype=np.float64)
    yeardays=np.asarray(index_slice.dayofyear, dtype=np.float64)
    return np.column_stack([
        np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
        np.sin(2*np.pi*weekdays/7), np.cos(2*np.pi*weekdays/7),
        np.sin(2*np.pi*yeardays/365.25), np.cos(2*np.pi*yeardays/365.25),
    ]).ravel().astype('float32')


def engineered_past(past, colnames=feature_cols):
    # Shape [lookback, n_features]. All input timestamps are strictly BEFORE forecast origin.
    x=past[:,0].astype('float64')
    features=[]
    for lag in (1,2,3,6,12,24,48,72,168):
        if lag <= len(x): features.append(float(x[-lag]))
    for w in (3,6,12,24,48,72,168):
        if w <= len(x):
            v=x[-w:]
            features += [float(v.mean()), float(v.std()), float(v.min()),float(v.max())]
    for w in (12,24,48,168):
        if w <= len(x):
            v=x[-w:]
            t=np.arange(w, dtype=float)
            features.append(float(np.dot(t-t.mean(),v-v.mean())/np.dot(t-t.mean(),t-t.mean())))
    for lag in (1,6,12,24,48,168):
        if lag <= len(x): features.append(float(x[-1]-x[-lag]))
    # This operation uses only historical sensor readings.
    for c in ['PM10','NO2','O3','TEMP','PRES','RAIN','WSPM']:
        v=past[:,colnames.index(c)]
        for w in (6,24,168):
            if w<=len(x):
                z=v[-w:]
                features.extend([float(z.mean()),float(z.std())])
    return np.asarray(features, dtype='float32')

# Forecast origins inside each partition, with target windows fully inside that partition.
all_origins = np.arange(LOOKBACK, len(frame)-HORIZON+1, dtype=int)
train_origins=all_origins[(all_origins+HORIZON <= train_end)] [::TRAIN_ORIGIN_STRIDE]
val_origins=all_origins[(all_origins >= train_end)&(all_origins+HORIZON <= val_end)] [::EVAL_ORIGIN_STRIDE]
test_origins=all_origins[(all_origins >= val_end)] [::EVAL_ORIGIN_STRIDE]

raw_values=frame[feature_cols].to_numpy(dtype='float32')
target_values=frame[TARGET].to_numpy(dtype='float32')

def make_windows(origins):
    X=np.stack([raw_values[t-LOOKBACK:t] for t in origins])
    F=np.stack([future_calendar(frame.index[t:t+HORIZON]) for t in origins])
    E=np.stack([engineered_past(raw_values[t-LOOKBACK:t]) for t in origins])
    Y=np.stack([target_values[t:t+HORIZON] for t in origins])
    M=np.stack([observed_pm25[t:t+HORIZON] for t in origins])
    return dict(X=X,F=F,E=E,Y=Y,M=M,origins=origins)

train=make_windows(train_origins)
val=make_windows(val_origins)
test=make_windows(test_origins)
# Supervised train/early-stopping labels must be real measurements, not imputed values.
train_complete=train['M'].all(axis=1)
val_complete=val['M'].all(axis=1)

def subset_rows(part,mask):
    return {k:v[mask] for k,v in part.items()}
train=subset_rows(train,train_complete)
val_fit=subset_rows(val,val_complete)
assert min(len(train['Y']),len(val_fit['Y']),len(test['Y'])) >= 10
print('Train/val/test forecast origins:',len(train['Y']),len(val['Y']),len(test['Y']))
print('Validation complete-label origins:',len(val_fit['Y']))
print('Raw sequence shape:',train['X'].shape,' statistics:',train['E'].shape,
      ' future calendar:',train['F'].shape)
assert np.max(train['origins']+HORIZON) <= train_end
assert np.min(val['origins']) >= train_end
assert np.min(test['origins']) >= val_end

## 5. Train-only normalization, quality checks, and a seasonal baseline

We standardize neural features and regression targets using **training data only**. CatBoost receives unscaled physical feature values and predicts the same standardized target. All predictions are inverse-transformed and clipped to nonnegative concentrations before scoring. **The primary metric is observed-only RMSE on held-out test timestamps.**

In [ ]:
x_mean=raw_values[:train_end].mean(axis=0)
x_std=raw_values[:train_end].std(axis=0).clip(1e-6)
y_mean=float(target_values[:train_end].mean())
y_std=max(float(target_values[:train_end].std()),1e-6)
e_mean=train['E'].mean(axis=0)
e_std=train['E'].std(axis=0).clip(1e-6)

for part in (train,val,val_fit,test):
    part['Xs']=(part['X']-x_mean)/x_std
    part['Es']=(part['E']-e_mean)/e_std
    part['Ys']=(part['Y']-y_mean)/y_std

# Explicit leakage check: every window's past ends strictly before its first target.
for part in (train,val,test):
    assert np.all(part['origins']-LOOKBACK>=0)
    assert np.all(part['origins']+HORIZON<=len(frame))


def observed_rmse(y_true,y_pred,observed):
    diff=np.asarray(y_pred,dtype=float)-np.asarray(y_true,dtype=float)
    valid=np.asarray(observed,dtype=bool)
    return float(np.sqrt(np.mean(diff[valid]**2)))


def score_predictions(name, val_pred, test_pred, seconds=0):
    val_pred=np.maximum(0,val_pred)
    test_pred=np.maximum(0,test_pred)
    results.append(dict(model=name, val_rmse=observed_rmse(val['Y'],val_pred,val['M']),
                        test_rmse=observed_rmse(test['Y'],test_pred,test['M']),
                        seconds=round(seconds,1)))
    predictions[name]=test_pred
    print(f"{name:23s} val RMSE={results[-1]['val_rmse']:7.2f}  test RMSE={results[-1]['test_rmse']:7.2f} µg/m³  ({seconds:.1f}s)")


def periodic_naive(part,period):
    # Repeat the most recent observed historic seasonal block (always in input window).
    last=part['X'][:,-period:,0]
    return np.tile(last,(1,int(np.ceil(HORIZON/period))))[:,:HORIZON]

results=[]; predictions={}
for name,period in [('Last-day naive',24),('Last-week naive',168)]:
    score_predictions(name,periodic_naive(val,period),periodic_naive(test,period))

## 6. Neural network architectures

GRU/LSTM compress the history into a recurrent hidden state; TCN uses causal dilated convolutions (receptive field ≥ 168); Transformer attends over all **past** hours with positional embeddings. All models are **direct multi-output** forecasters, with the same 24-hour (or 168-hour) target horizon, and the same known future-calendar covariates. The MLP is included as a deliberately strong non-sequential baseline.

In [ ]:
class ForecastHead(nn.Module):
    def __init__(self, encoder_width, future_width, extra_width, horizon):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(encoder_width+future_width+extra_width,128),
                               nn.ReLU(),nn.Dropout(.1),nn.Linear(128,horizon))
    def forward(self,encoded,calendar,extra):
        return self.net(torch.cat([encoded,calendar,extra],dim=1))

class MLPForecast(nn.Module):
    def __init__(self,n_features,f_width,e_width):
        super().__init__()
        self.encoder=nn.Sequential(nn.Flatten(),nn.Linear(LOOKBACK*n_features,192),
                                   nn.ReLU(),nn.Dropout(.12),nn.Linear(192,96),nn.ReLU())
        self.head=ForecastHead(96,f_width,e_width,HORIZON)
    def forward(self,x,f,e): return self.head(self.encoder(x),f,e)

class RNNForecast(nn.Module):
    def __init__(self,n_features,f_width,e_width,kind='GRU'):
        super().__init__()
        klass=nn.GRU if kind=='GRU' else nn.LSTM
        self.rnn=klass(input_size=n_features,hidden_size=64,batch_first=True,num_layers=1)
        self.head=ForecastHead(64,f_width,e_width,HORIZON)
    def forward(self,x,f,e):
        h,_=self.rnn(x)
        return self.head(h[:,-1,:],f,e)

class CausalResidual(nn.Module):
    def __init__(self,channels,dilation):
        super().__init__()
        self.pad=2*dilation
        self.conv=nn.Conv1d(channels,channels,kernel_size=3,dilation=dilation,padding=self.pad)
        self.norm=nn.GroupNorm(4,channels)
        self.drop=nn.Dropout(.10)
    def forward(self,x):
        h=self.conv(x)[:,:,:x.shape[-1]]
        return x+self.drop(torch.relu(self.norm(h)))

class TCNForecast(nn.Module):
    def __init__(self,n_features,f_width,e_width):
        super().__init__()
        self.start=nn.Conv1d(n_features,48,1)
        self.blocks=nn.Sequential(*[CausalResidual(48,d) for d in (1,2,4,8,16,32,64)])
        self.head=ForecastHead(48,f_width,e_width,HORIZON)
    def forward(self,x,f,e):
        h=self.blocks(self.start(x.transpose(1,2)))[:,:,-1]
        return self.head(h,f,e)

class TransformerForecast(nn.Module):
    def __init__(self,n_features,f_width,e_width):
        super().__init__()
        d=48
        self.project=nn.Linear(n_features,d)
        self.pos=nn.Parameter(torch.randn(1,LOOKBACK,d)*0.02)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=4,dim_feedforward=128,
                                         dropout=.1,batch_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,num_layers=2)
        self.head=ForecastHead(d*2,f_width,e_width,HORIZON)
    def forward(self,x,f,e):
        h=self.transformer(self.project(x)+self.pos)
        return self.head(torch.cat([h[:,-1,:],h.mean(dim=1)],dim=1),f,e)


def construct_model(name):
    dimensions=(len(feature_cols),train['F'].shape[1], train['E'].shape[1] if NEURAL_USE_ENGINEERED else 0)
    if name=='MLP': return MLPForecast(*dimensions)
    if name in ('GRU','LSTM'): return RNNForecast(*dimensions,kind=name)
    if name=='TCN': return TCNForecast(*dimensions)
    if name=='Transformer': return TransformerForecast(*dimensions)
    raise ValueError(name)

## 7. Train the five neural models

Validation RMSE (only actual observations) determines early stopping. The same fully observed training windows are used for CatBoost and for every neural model; we do not use the test set for training decisions. Set `QUICK_RUN=False` for a more meaningful training budget.

In [ ]:
def loader(part,shuffle=False):
    empty=np.zeros((len(part['Y']),0),dtype='float32')
    extras=part['Es'] if NEURAL_USE_ENGINEERED else empty
    tensors=[torch.from_numpy(np.asarray(part[k],dtype='float32')) for k in ('Xs','F')]
    tensors += [torch.from_numpy(extras.astype('float32')),
                torch.from_numpy(part['Ys'].astype('float32')),
                torch.from_numpy(part['M'].astype('float32'))]
    return DataLoader(TensorDataset(*tensors),batch_size=BATCH_SIZE,shuffle=shuffle)

train_loader=loader(train,True)
val_loader=loader(val)

@torch.no_grad()
def nn_predict(model,part):
    model.eval()
    out=[]
    for x,f,e,_,_ in loader(part):
        out.append(model(x.to(DEVICE),f.to(DEVICE),e.to(DEVICE)).cpu().numpy())
    return np.concatenate(out)*y_std+y_mean


def fit_neural(name):
    start=time.time()
    model=construct_model(name).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    best=np.inf; best_state=None; stale=0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for x,f,e,y,mask in train_loader:
            x,f,e,y,mask=[a.to(DEVICE) for a in (x,f,e,y,mask)]
            optimizer.zero_grad(set_to_none=True)
            pred=model(x,f,e)
            loss=((pred-y)**2*mask).sum()/mask.sum().clamp_min(1)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step()
        # Evaluate observed-only, in ORIGINAL concentration units
        valid_pred=nn_predict(model,val)
        valid_err=observed_rmse(val['Y'],np.maximum(0,valid_pred),val['M'])
        if valid_err < best-1e-4:
            best=valid_err; best_state=copy.deepcopy(model.state_dict()); stale=0
        else:
            stale+=1
            if stale>=PATIENCE: break
    model.load_state_dict(best_state)
    score_predictions(name,nn_predict(model,val),nn_predict(model,test),time.time()-start)
    return model

neural_models={}
for name in RUN_MODELS:
    neural_models[name]=fit_neural(name)

## 8. CatBoost: identical windows, with and without statistical features

Both CatBoost versions use exactly the same chronological train/validation/test origins, future calendar, target scaling and hyperparameter budget. The **only difference is** the extra engineered past-statistic columns. CatBoost uses its own validation early stopping; test data stay untouched.

In [ ]:
def catboost_features(part, engineered=False):
    x=np.concatenate([part['X'].reshape(len(part['X']),-1),part['F']],axis=1)
    if engineered: x=np.concatenate([x,part['E']],axis=1)
    return x.astype('float32')


def fit_catboost(name, engineered):
    start=time.time()
    model=CatBoostRegressor(loss_function='MultiRMSE',iterations=CB_ITERATIONS,
                            learning_rate=.055,depth=6,l2_leaf_reg=5,
                            random_seed=SEED,thread_count=4,verbose=False)
    model.fit(catboost_features(train,engineered),train['Ys'],
              eval_set=(catboost_features(val_fit,engineered),val_fit['Ys']),
              early_stopping_rounds=CB_EARLY_STOPPING,verbose=False)
    val_pred=np.asarray(model.predict(catboost_features(val,engineered)))*y_std+y_mean
    test_pred=np.asarray(model.predict(catboost_features(test,engineered)))*y_std+y_mean
    score_predictions(name,val_pred,test_pred,time.time()-start)
    print('   best tree iteration:',model.best_iteration_)
    return model

catboost_raw=fit_catboost('CatBoost raw',False)
catboost_engineered=fit_catboost('CatBoost engineered',True)

## 9. Ranking, per-horizon errors, and visual examples

Rank by **observed-only test RMSE**, not validation error. Validation is used to choose training iterations and epochs, so it is optimistic for the selected models. The test set measures forecasts on a later, unseen time period.

In [ ]:
ranking=pd.DataFrame(results).sort_values('test_rmse').reset_index(drop=True)
display(ranking.style.format({'val_rmse':'{:.2f}','test_rmse':'{:.2f}','seconds':'{:.1f}'}))

ax=ranking.set_index('model')['test_rmse'].sort_values().plot(kind='barh',figsize=(9,4),title='Observed-only held-out PM2.5 forecast RMSE (lower is better)')
ax.set_xlabel('µg/m³'); plt.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(10,4))
for name in ['MLP','GRU','LSTM','TCN','Transformer','CatBoost raw','CatBoost engineered']:
    if name not in predictions: continue
    error=(predictions[name]-test['Y'])**2
    mask=test['M']
    horizon_rmse=np.sqrt((error*mask).sum(axis=0)/np.maximum(mask.sum(axis=0),1))
    ax.plot(np.arange(1,HORIZON+1),horizon_rmse,label=name,alpha=.85)
ax.set(title='RMSE by forecast lead time',xlabel='Hours ahead',ylabel='RMSE (µg/m³)')
ax.legend(ncol=2,fontsize=8); plt.tight_layout(); plt.show()

example_id=min(5,len(test['Y'])-1)
actual=test['Y'][example_id].astype(float).copy()
actual[~test['M'][example_id]]=np.nan
h=np.arange(1,HORIZON+1)
fig,ax=plt.subplots(figsize=(11,4))
ax.plot(h,actual,color='black',lw=2,label='Actually observed PM2.5')
for name in ranking['model'].head(4):
    ax.plot(h,predictions[name][example_id],label=name,alpha=.85)
ax.set(xlabel='Hours ahead',ylabel='PM2.5 µg/m³',title=f"One held-out forecast starting {frame.index[test['origins'][example_id]]}")
ax.legend();plt.tight_layout();plt.show()

out=Path('beijing_forecast_results');out.mkdir(exist_ok=True)
ranking.to_csv(out/'ranking.csv',index=False)
print('Saved:',out/'ranking.csv')

## 10. Interpretation and extension ideas

- **If CatBoost wins:** that's a legitimate result. Longer horizons and more data may favor sequence models, but there is no universal superiority. Try `HORIZON=168` and tune each architecture using **validation only**.
- **If GRU, LSTM, TCN or Transformer wins:** check that advantage repeats over multiple random seeds and multiple chronological test blocks. A win in one random split is weak evidence.
- **Fairness:** raw CatBoost and all neural models can access the same original historical covariates and deterministic future calendar. Engineered CatBoost is a separate stronger feature pipeline; setting `NEURAL_USE_ENGINEERED=True` tests the neural models with the same additional statistics.
- **No future weather leakage:** future meteorological observations are intentionally excluded. An operational weather forecast would need a separate forecast source.
- **Missing values:** test labels imputed solely as modeling placeholders are excluded from the primary metric. The training/validation supervised windows used to fit CatBoost have fully observed labels.
- **Computational budget:** equal raw information is not equal tuning effort. Run the full budget and try a comparable validation search over depth/learning rate/regularization for CatBoost, and hidden size/lr/weight decay/early stopping for neural models before drawing strong conclusions.
- **More data:** to give sequence models a more compelling test, train one shared model on several stations with station embedding, then evaluate on held-out future weeks of the **same stations**. Keep station IDs and dates explicit; don't randomly intermix overlapping target windows.

**Source:** Chen, S. (2017). Beijing Multi-Site Air Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5RK5G.

In [ ]:
# W&B experiment tracking (optional; runs after saved outputs exist)
# This cell can be run separately after the notebook has saved comparison.csv.
# It does NOT retrain any models and does not modify local result files.
# Requires: pip install wandb && wandb login
USE_WANDB = False  # Switch to True to publish one W&B run per saved model.
import sys
from pathlib import Path
roots = [Path.cwd(), Path.cwd() / "Time_Series_Benchmark_Final", Path.cwd().parent,
         Path.cwd().parent / "Time_Series_Benchmark_Final"]
tracking_root = next((p for p in roots if (p / "wandb_tracking.py").exists()), None)
if tracking_root is None:
    print("Optional W&B helper not found: put wandb_tracking.py beside the benchmark notebooks.")
else:
    if str(tracking_root.resolve()) not in sys.path:
        sys.path.insert(0, str(tracking_root.resolve()))
    from wandb_tracking import log_comparison_csv, summarize_comparison
    print(summarize_comparison('beijing_forecast_results', dataset='beijing', csv_name='ranking.csv'))
    log_comparison_csv('beijing_forecast_results', dataset='beijing', csv_name='ranking.csv', enabled=USE_WANDB)
